In [ ]:
# -*- coding: utf-8 -*-

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')

# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [2]:
# --- Classes de mida de l'incendi (Q4) ---
classes_mida = ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
df['ClasseMida'] = pd.cut(
    df['SuperficieTotal'],
    bins=[-0.01, 1, 10, 100, 500, 1e9],
    labels=classes_mida
)
mida_scale = alt.Scale(
    domain=classes_mida,
    scheme=alt.SchemeParams(name='inferno', extent=[0.12, 0.92]),
    reverse=True
)

In [4]:
# %% ===================================================================
# Q4 FINAL (trienni) · Hectàrees cremades (barra mirall) + Superfície Forestal (donut)
#                       + Nombre d'incendis per magnitud (boles)
# Versió 2: llegenda única, títols alineats, barra x1.6, donut amb angle
#           de sortida personalitzat, boles amb colors consistents.
# =======================================================================
# SUPOSICIONS: aquestes variables ja existeixen de cel·les anteriors:
#   - df             -> 'Any', 'SuperficieTotal', 'ClasseMida',
#                        'SuperficieTotalForestal', 'SuperficieTotalNoForestal',
#                        'Nom municipi', 'Nom comarca'
#   - classes_mida   -> ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
#   - mida_scale     -> alt.Scale(domain=classes_mida, scheme=..., reverse=True)
# -----------------------------------------------------------------------
import math
import pandas as pd
import altair as alt

# ---------------------------------------------------------------------
# 0) TRIENNIS
# ---------------------------------------------------------------------
ANY_MIN = int(df['Any'].min())
MIDA_PERIODE = 3


def assigna_trienni(any_valor, any_min=ANY_MIN, mida=MIDA_PERIODE):
    idx = (any_valor - any_min) // mida
    inici = any_min + idx * mida
    fi = inici + mida - 1
    return f"{inici}-{fi}"


df['PeriodeQ4'] = df['Any'].apply(assigna_trienni)
labels_q4 = sorted(df['PeriodeQ4'].unique(), key=lambda p: int(p.split('-')[0]))

# ---------------------------------------------------------------------
# 1) HECTÀREES CREMADES per classe i trienni — ORDRE DESCENDENT
#    (la classe >500 ha toca el 0/centre; les petites arriben a la punta)
# ---------------------------------------------------------------------
classes_mida_desc = list(reversed(classes_mida))

df_q4_mida_raw = (
    df.groupby(['PeriodeQ4', 'ClasseMida'], as_index=False, observed=True)
    .agg(Superficie=('SuperficieTotal', 'sum'), NumFocs=('SuperficieTotal', 'count'))
    .rename(columns={'PeriodeQ4': 'Periode'})
)

parts_abs = []
for p in labels_q4:
    sub = (
        df_q4_mida_raw[df_q4_mida_raw['Periode'] == p]
        .set_index('ClasseMida').reindex(classes_mida_desc).reset_index()
    )
    sub['Superficie'] = sub['Superficie'].fillna(0)
    sub['NumFocs'] = sub['NumFocs'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['pct'] = sub['Superficie'] / tot if tot else 0
    sub['x1_abs'] = sub['Superficie'].cumsum()
    sub['x0_abs'] = sub['x1_abs'] - sub['Superficie']
    sub['x_mig_abs'] = (sub['x0_abs'] + sub['x1_abs']) / 2
    sub['Periode'] = p
    sub['Total_Periode'] = tot
    sub['label_pct'] = sub['pct'].apply(lambda v: f"{round(v * 100)}%" if v >= 0.05 else "")
    parts_abs.append(sub)
df_q4_mida_abs = pd.concat(parts_abs, ignore_index=True)

max_ha_periode = df_q4_mida_abs.groupby('Periode')['x1_abs'].max().max()
totals_periode = df_q4_mida_abs.groupby('Periode', as_index=False)['Total_Periode'].first()
totals_periode['label_total'] = totals_periode['Total_Periode'].apply(lambda v: f"{v:,.0f} ha")

# 0 a la dreta (tocant el trienni), els valors s'allunyen cap a l'esquerra.
# Format sense notació científica.
X_SCALE_ABS = alt.Scale(domain=[max_ha_periode * 1.1, 0], nice=False)

# ---------------------------------------------------------------------
# 2) SUPERFÍCIE FORESTAL vs NO FORESTAL — donut amb angle de sortida
#    personalitzat: Forestal comença a la dreta (3 en punt) i avança en
#    sentit horari; No Forestal queda, doncs, amunt a l'esquerra.
# ---------------------------------------------------------------------
df_q4_tipus_raw = (
    df.groupby('PeriodeQ4', as_index=False, observed=True)
    .agg(Forestal=('SuperficieTotalForestal', 'sum'), NoForestal=('SuperficieTotalNoForestal', 'sum'))
    .rename(columns={'PeriodeQ4': 'Periode'})
    .melt(id_vars='Periode', value_vars=['Forestal', 'NoForestal'], var_name='Tipus', value_name='Superficie')
)
tipus_domain = ['Forestal', 'NoForestal']
tipus_scale = alt.Scale(domain=tipus_domain, range=['#1b7837', '#d73027'])  # verd / vermell

OFFSET_RAD = 0

parts_tipus = []
for p in labels_q4:
    sub = df_q4_tipus_raw[df_q4_tipus_raw['Periode'] == p].set_index('Tipus').reindex(tipus_domain).reset_index()
    sub['Superficie'] = sub['Superficie'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['frac'] = sub['Superficie'] / tot if tot else 0
    sub['cum_end'] = sub['frac'].cumsum()
    sub['cum_start'] = sub['cum_end'] - sub['frac']
    sub['theta0'] = OFFSET_RAD + sub['cum_start'] * 2 * math.pi
    sub['theta1'] = OFFSET_RAD + sub['cum_end'] * 2 * math.pi
    sub['theta_mid'] = (sub['theta0'] + sub['theta1']) / 2
    sub['Periode'] = p
    sub['label_abs'] = sub.apply(
        lambda r: f"{r['frac']:,.0f} %" if r['Tipus'] == 'Forestal' else "", axis=1
    )
    parts_tipus.append(sub)
df_q4_tipus = pd.concat(parts_tipus, ignore_index=True)

# ---------------------------------------------------------------------
# 3) BOLES "RESTA" (<500 ha, 1 bola = 20 focs) — com a Q4c(v2):
#    no es toquen, espai mínim, màx. 6 files; 20 columnes (22 si cal)
# ---------------------------------------------------------------------
GROUP_SIZE_Q4 = 20
classes_resta = [c for c in classes_mida if c != '>500 ha']


def compta_boles_resta(subdf):
    total = 0
    for classe in classes_resta:
        n = int((subdf['ClasseMida'] == classe).sum())
        if n > 0 and classe == '100-500 ha':
            total += max(1, int(round(n / GROUP_SIZE_Q4)))
        else:
            total += int(n // GROUP_SIZE_Q4)
    return total


max_boles_resta = max(compta_boles_resta(df[df['PeriodeQ4'] == p]) for p in labels_q4)
COLS_RESTA_Q4 = 20 if max_boles_resta <= 20 * 6 else 22  # prova primer amb 20


def boles_resta_periode(subdf, periode):
    files = []
    idx_global = 0
    for classe in reversed(classes_resta):
        n = int((subdf['ClasseMida'] == classe).sum())
        if n > 0 and classe == '100-500 ha':
            n_boles = max(1, int(round(n / GROUP_SIZE_Q4)))
        else:
            n_boles = int(n // GROUP_SIZE_Q4)
        for _ in range(n_boles):
            files.append({
                'Periode': periode, 'ClasseMida': classe,
                'col': idx_global % COLS_RESTA_Q4, 'fila': idx_global // COLS_RESTA_Q4,
                'Incendis_Reals': n,
                'Equivalencia': f"1 bola = {GROUP_SIZE_Q4} incendis ({classe})"
            })
            idx_global += 1
    return pd.DataFrame(files)


df_boles_resta = pd.concat(
    [boles_resta_periode(df[df['PeriodeQ4'] == p], p) for p in labels_q4], ignore_index=True
)
max_fila_resta = max(int(df_boles_resta['fila'].max()), 2)

# ---------------------------------------------------------------------
# 4) BOLES GIF individuals (>500 ha) — MATEIX color que la resta (mida_scale),
#    alineades, espai entre columnes reduït. Màx. 4 columnes.
# ---------------------------------------------------------------------
COLS_GIF_Q4 = 4


def boles_gif_individuals(subdf, periode):
    files = []
    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for i, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        files.append({
            'Periode': periode, 'ClasseMida': '>500 ha',
            'col': i % COLS_GIF_Q4, 'fila': i // COLS_GIF_Q4,
            'Nom_Comarca': r['Nom comarca'], 'Ubicacio': f"{mun} ({int(r['Any'])})",
            'Any': int(r['Any']), 'ha_real': float(r['SuperficieTotal'])
        })
    return pd.DataFrame(files)


df_boles_gif = pd.concat(
    [boles_gif_individuals(df[df['PeriodeQ4'] == p], p) for p in labels_q4], ignore_index=True
)
max_fila_gif = max(int(df_boles_gif['fila'].max()) if len(df_boles_gif) else 2, 2)

# ---------------------------------------------------------------------
# 5) AMPLADES — barra x1.6 i boles GIF més compactes
# ---------------------------------------------------------------------
ALCADA_BARRA = 46
W_BAR_ABS = 750      
W_ANY_Q4 = 78
W_DONUT = 150
W_GIF = 100            # més estret -> boles GIF més juntes entre elles
W_RESTA = 260          # espai perquè 20-22 columnes no es toquin
ESPAI = 8
W_BOLES_COMBO = W_GIF + W_RESTA + ESPAI
TOTAL_WIDTH = W_BAR_ABS + ESPAI + W_ANY_Q4 + ESPAI + W_DONUT + ESPAI + W_BOLES_COMBO

ESTIL_TITOL = dict(fontSize=13, fontWeight='bold', align='center', color='#111111')


def _spacer(width, height=1):
    return alt.Chart(pd.DataFrame({'t': ['']})).mark_text().encode(text='t:N').properties(width=width, height=height)


# ---------------------------------------------------------------------
# 6) LLEGENDA ÚNICA (5 classes), construïda a mà per evitar que una
#    llegenda d'Altair "desbordi" la seva columna i desalineï la resta.
# ---------------------------------------------------------------------
leg_df = pd.DataFrame({'ClasseMida': classes_mida, 'x': range(len(classes_mida))})
base_leg = alt.Chart(leg_df).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[-0.6, len(classes_mida) - 0.3], nice=False)),
    y=alt.value(10)
)
leg_dots = base_leg.mark_circle(size=140).encode(color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None))
leg_text = base_leg.mark_text(align='left', dx=9, fontSize=11, color='#222222').encode(text='ClasseMida:N')
llegenda_unica = (leg_dots + leg_text).properties(width=TOTAL_WIDTH, height=22)

# ---------------------------------------------------------------------
# 7) CAPÇALERA DE TÍTOLS — mateixa tipografia, centrats sobre el seu
#    gràfic, tots a la mateixa fila
# ---------------------------------------------------------------------
tit_bar = alt.Chart(pd.DataFrame({'t': ["Hectàrees cremades segons magnitud de l'incendi"]})).mark_text(
    **ESTIL_TITOL
).encode(text='t:N').properties(width=W_BAR_ABS, height=1)

tit_any = alt.Chart(pd.DataFrame({'t': ['Trieni']})).mark_text(**ESTIL_TITOL).encode(text='t:N').properties(
    width=W_ANY_Q4, height=1
)

tit_donut = alt.Chart(pd.DataFrame({'t': ['Superfície Forestal cremada']})).mark_text(**ESTIL_TITOL).encode(
    text='t:N'
).properties(width=W_DONUT, height=1)

tit_boles = alt.Chart(pd.DataFrame({'t': ["Nombre d'incendis per magnitud"]})).mark_text(**ESTIL_TITOL).encode(
    text='t:N'
).properties(width=W_BOLES_COMBO, height=1)

fila_titols = alt.hconcat(tit_bar, tit_any, tit_donut, tit_boles, spacing=ESPAI)

# ---------------------------------------------------------------------
# 8) UNA FILA PER TRIENNI
# ---------------------------------------------------------------------
files_final = [llegenda_unica, fila_titols]

for idx_p, p in enumerate(labels_q4):
    is_last = idx_p == len(labels_q4) - 1

    sub_bar = df_q4_mida_abs[df_q4_mida_abs['Periode'] == p]
    total_p_df = totals_periode[totals_periode['Periode'] == p]

    c_bar = (
        alt.Chart(sub_bar).mark_bar(height=ALCADA_BARRA).encode(
            x=alt.X(
                'x0_abs:Q', scale=X_SCALE_ABS,
                axis=alt.Axis(title='Superfície total cremada (ha)', format=',.0f') if is_last else None
            ),
            x2='x1_abs:Q',
            color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
            tooltip=[
                alt.Tooltip('Periode:N', title='Trienni'),
                alt.Tooltip('ClasseMida:N', title='Magnitud'),
                alt.Tooltip('Superficie:Q', title='Hectàrees cremades (ha)', format=',.1f'),
                alt.Tooltip('NumFocs:Q', title="Nombre d'incendis", format=',')
            ]
        )
        + alt.Chart(sub_bar[sub_bar['pct'] >= 0.05]).mark_text(
            color='white', fontSize=10.5, fontWeight='bold', baseline='middle'
        ).encode(x=alt.X('x_mig_abs:Q', scale=X_SCALE_ABS), text='label_pct:N')
        + alt.Chart(total_p_df).mark_text(
            align='right', dx=-6, fontSize=12, fontWeight='bold', color='#000000', baseline='middle'
        ).encode(x=alt.X('Total_Periode:Q', scale=X_SCALE_ABS), text='label_total:N')
    ).properties(width=W_BAR_ABS, height=ALCADA_BARRA + 8)

    # --- Trienni (centre) ---
    c_any = alt.Chart(pd.DataFrame({'t': [p]})).mark_text(
        fontSize=16, fontWeight='bold', align='center', color='#111111'
    ).encode(text='t:N').properties(width=W_ANY_Q4, height=ALCADA_BARRA + 8)

    # --- Donut Forestal / No Forestal ---
    sub_t = df_q4_tipus[df_q4_tipus['Periode'] == p]
    arc = alt.Chart(sub_t).mark_arc(innerRadius=8, outerRadius=28, stroke='white', strokeWidth=1).encode(
        theta=alt.Theta('theta0:Q', scale=None),
        theta2=alt.Theta2('theta1:Q'),
        color=alt.Color('Tipus:N', scale=tipus_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('Tipus:N', title='Cobertura'),
            alt.Tooltip('Superficie:Q', title='Hectàrees (ha)', format=',.1f'),
            alt.Tooltip('frac:Q', title='Percentatge', format='.1%')
        ]
    )
    text_arc = alt.Chart(sub_t[sub_t['label_abs'] != ""]).mark_text(
        radius=24, fontSize=10, fontWeight='bold', color='white'
    ).encode(theta=alt.Theta('theta_mid:Q', scale=None), text='label_abs:N')
    c_donut = (arc + text_arc).properties(width=W_DONUT, height=ALCADA_BARRA + 8)

    # --- Boles GIF individuals (mateix color que '>500 ha') ---
    sub_g = df_boles_gif[df_boles_gif['Periode'] == p]
    c_boles_gif = alt.Chart(sub_g).mark_circle(
        size=95, opacity=0.95, stroke='white', strokeWidth=0.5
    ).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.5, COLS_GIF_Q4 - 0.5], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.5, max_fila_gif + 0.5], reverse=True, nice=False), axis=None),
        color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
        tooltip=[
            alt.Tooltip('Ubicacio:N', title='Gran Incendi Forestal (GIF)'),
            alt.Tooltip('Nom_Comarca:N', title='Comarca'),
            alt.Tooltip('Any:Q', title='Any'),
            alt.Tooltip('ha_real:Q', title='Hectàrees cremades (ha)', format=',.1f')
        ]
    ).properties(width=W_GIF, height=ALCADA_BARRA + 8)

    # --- Boles "resta" agrupades (no es toquen, estil Q4c v2) ---
    sub_r = df_boles_resta[df_boles_resta['Periode'] == p]
    c_boles_resta = alt.Chart(sub_r).mark_circle(
        size=42, opacity=0.95, stroke='white', strokeWidth=0.4
    ).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-1, COLS_RESTA_Q4], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.8, max_fila_resta + 0.8], reverse=True, nice=False), axis=None),
        color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Trienni'),
            alt.Tooltip('ClasseMida:N', title='Categoria'),
            alt.Tooltip('Incendis_Reals:Q', title="Total incendis d'aquesta classe", format=','),
            alt.Tooltip('Equivalencia:N', title='Escala')
        ]
    ).properties(width=W_RESTA, height=ALCADA_BARRA + 8)

    files_final.append(
        alt.hconcat(c_bar, c_any, c_donut, c_boles_gif, c_boles_resta, spacing=ESPAI)
        .resolve_scale(color='independent')
    )

# ---------------------------------------------------------------------
# 9) LLEGENDA Forestal / No Forestal, alineada sota el donut
# ---------------------------------------------------------------------
lleg_tipus_df = pd.DataFrame({
    'Tipus': ['Forestal', 'NoForestal'],
    'label': ['Forestal', 'No Forestal'],
    'y': [0, 1]
})
base_t = alt.Chart(lleg_tipus_df).encode(
    y=alt.Y('y:Q', axis=None, scale=alt.Scale(domain=[-0.6, 1.6]))
)
dot_t = base_t.mark_circle(size=90).encode(
    x=alt.value(10),
    color=alt.Color('Tipus:N', scale=tipus_scale, legend=None)
)
text_t = base_t.mark_text(align='left', fontSize=10.5, color='#222222').encode(
    x=alt.value(20), text='label:N'
)

llegenda_tipus_sota = (dot_t + text_t).properties(width=W_DONUT, height=32)

fila_llegenda_tipus = alt.hconcat(
    _spacer(W_BAR_ABS, 32), _spacer(W_ANY_Q4, 32), llegenda_tipus_sota, _spacer(W_BOLES_COMBO, 32), spacing=ESPAI
)
files_final.append(fila_llegenda_tipus)


q4_final_trienni = alt.vconcat(*files_final, spacing=3).properties(
    title=alt.TitleParams(
        "Q4 · Evolució de la distribució dels incendis forestals a Catalunya per trienni (1998–2024)",
        fontSize=16, anchor='start'
    )
).configure_view(strokeWidth=0).configure_axis(grid=False)

q4_final_trienni.save('outputs/Q4.json')

# q4_final_trienni.save('outputs/Q4V2.html')
